# Playground — Step 6: Family 3 Concept Schedulers

Family 3 keeps single-concept scoring and varies **which** concept is active over time:

| scheduler | behavior |
|---|---|
| `RoundRobin()` | F3.a — cycle through concepts, one per step |
| `Stochastic(probs, seed)` | F3.b — sample the active concept per step (seeded, reproducible) |
| `SentenceBoundary(tokenizer)` | F3.c — advance concept at sentence boundaries (per input) |

They plug into the same `scorer` seam as the aggregators, via a stateful protocol the
loops honor (`reset` at generation start, `observe` per step). The loop's `weights` are
ignored — one concept at a time is the point.

**Why Family 3 matters:** concepts that can never co-occur in one word can co-occur in a
passage. Schedulers are the control condition separating "the method failed" from
"language doesn't have that word".

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))  # repo root, so `frames` imports work

In [ ]:
import torch

from frames.representations import FrameUnembeddingRepresentation
from frames.representations.schedulers import RoundRobin, SentenceBoundary, Stochastic

In [ ]:
fur = FrameUnembeddingRepresentation.from_model_id(
    "hugging-quants/Meta-Llama-3.1-8B-Instruct-AWQ-INT4",
    device_map="cuda:0",
    torch_dtype=torch.float16,
)

MIN_LEMMAS = 11
MAX_TOKENS = 3
K = 3
STEPS = 24  # a bit longer, so schedules have room to act


def chat(user: str, assistant: str = "") -> str:
    return (
        f"<|start_header_id|>user<|end_header_id|>{user}<|eot_id|>"
        f"<|start_header_id|>assistant<|end_header_id|>{assistant}"
    )


def answer(text: str) -> str:
    return text.split("assistant<|end_header_id|>")[-1]


PROMPTS = [
    chat("Tell me a short story.", "Once"),
    chat("Describe your ideal weekend.", "My ideal weekend starts"),
]

joy = fur.get_concept_cached("joy.n.01", MIN_LEMMAS, MAX_TOKENS)
dog = fur.get_concept_cached("dog.n.01", MIN_LEMMAS, MAX_TOKENS)
CONCEPTS = [joy, dog]

## 1 — Round-robin: alternate joy/dog every step

In [ ]:
texts_rr, probe_rr = fur.generate_with_topk_multi_guide(
    PROMPTS, guides=CONCEPTS, k=K, steps=STEPS, scorer=RoundRobin(),
)
for t in texts_rr:
    print(answer(t)[:150], "\n")

## 2 — Stochastic: 70% joy / 30% dog, seeded

Run the cell twice — the schedule (and output) is identical thanks to `reset` re-seeding.

In [ ]:
texts_st, _ = fur.generate_with_topk_multi_guide(
    PROMPTS, guides=CONCEPTS, k=K, steps=STEPS,
    scorer=Stochastic([0.7, 0.3], seed=42),
)
texts_st_again, _ = fur.generate_with_topk_multi_guide(
    PROMPTS, guides=CONCEPTS, k=K, steps=STEPS,
    scorer=Stochastic([0.7, 0.3], seed=42),
)
assert texts_st == texts_st_again, "seeded schedule must be reproducible"
print("reproducibility check passed\n")
for t in texts_st:
    print(answer(t)[:150], "\n")

## 3 — Sentence boundary: one concept per sentence

Each input advances its own concept counter when its representative beam emits
sentence-ending punctuation.

In [ ]:
texts_sb, _ = fur.generate_with_topk_multi_guide(
    PROMPTS, guides=CONCEPTS, k=K, steps=STEPS,
    scorer=SentenceBoundary(fur.model.tokenizer),
)
for t in texts_sb:
    print(answer(t)[:200], "\n")

## 4 — Side-by-side: simultaneous (F2.a) vs scheduled (F3)

Same concepts, same budget — composition in score space vs in time.

In [ ]:
texts_sum, _ = fur.generate_with_topk_multi_guide(
    PROMPTS, guides=CONCEPTS, k=K, steps=STEPS, normalize="zscore",
)

for sum_t, rr_t, sb_t in zip(texts_sum, texts_rr, texts_sb):
    print("F2.a sum+zscore:", answer(sum_t)[:120])
    print("F3.a round-robin:", answer(rr_t)[:120])
    print("F3.c per-sentence:", answer(sb_t)[:120])
    print()

## 5 — Schedulers under beam search

The protocol works in both loops.

In [ ]:
texts_beam_rr, _ = fur.generate_with_topk_beam_guide(
    PROMPTS, concepts=CONCEPTS, k=K, steps=STEPS, scorer=RoundRobin(),
)
for t in texts_beam_rr:
    print(answer(t)[:150], "\n")